# 20b · Event subsets by attention

Inputs: `panel_event`, `placebo_draws_strict`

Outputs: `out20b/`, zipped as `out20b.zip`: `est_20b_subsets.csv`, `est_20b_calendar.csv`, `est_20b_levels.csv`, `fig_20b_calendar.png`, `build_report_20b.json`

Pooled L − C estimates for event subsets defined by attention (in 03, Willow and Google's white paper are the two main events with significant attention shocks). Estimator and inference of 20.

| Subset | Events |
|---|---|
| A | Willow |
| B | Google white paper |
| C | Willow + Google (high attention) |
| D | the five events without BIP-361 |
| E | Gidney, Lopp, BIP-360, BIP-361 (low attention) |
| F | all six (as in 20) |

Also exported for the supplementary materials: per-date θ, baseline and placebo band for the calendar-time figure (`est_20b_calendar.csv`), and treated and control level paths of the main events for the Kaplan–Meier figure (`est_20b_levels.csv`).

In [ ]:
# Configuration
PROJECT_ID = "your-gcp-project"
DATASET    = "qbtc_raw"
LOCATION   = "US"
AUTO_RUN_USD, PRICE_PER_TIB_USD = 5.0, 6.25

H_MAIN      = 90                     # local linear bandwidth (days, standard deviation of the Gaussian kernel)
BANDWIDTHS  = [60, 90, 120]
POOLS       = ["main", "sym30"]      # the two pools of placebo_draws_strict; main is primary
REPORT_DAYS = (7, 30)
OUT_C = ["y1_strict", "y1_mid", "y1_loose", "y2", "y0", "y3", "y1_strict_held", "y3_held"]   # cumulative shares
OUT_D = ["y1_strict", "y1_strict_held", "y3", "y0"]                                          # daily shares
HZ    = ["y1s_50", "y1s_gt0", "y1sh_50", "y0_50"]                                            # hazard events
SUBSETS = {"A_willow": ["willow"], "B_google": ["google"], "C_high_attention": ["willow", "google"],
           "D_no_bip361": ["willow", "gidney", "lopp", "bip360", "google"],
           "E_low_attention": ["gidney", "lopp", "bip360", "bip361"],
           "F_all_six": ["willow", "gidney", "lopp", "bip360", "google", "bip361"]}
N_DRAWS, SEED = 1000, 20261001
REF_20_POOLED = 8.184148315025512e-05   # pooled effect of the main specification in 20 (LC, y1 strict, key-weighted, +30, main pool, h = 90)
OUT_DIR = "out20b"

try:
    from google.colab import auth
    auth.authenticate_user()
except ImportError:
    pass
from google.cloud import bigquery
import json, os, zipfile, datetime as dt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

client = bigquery.Client(project=PROJECT_ID, location=LOCATION)
DS = f"{PROJECT_ID}.{DATASET}"
SPENT = {"usd": 0.0}
os.makedirs(OUT_DIR, exist_ok=True)

def q(sql):
    job = client.query(sql, job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False))
    usd = job.total_bytes_processed / 2**40 * PRICE_PER_TIB_USD
    assert usd < AUTO_RUN_USD, f"estimated ${usd:.2f}, above the auto-run limit"
    SPENT["usd"] += usd
    return client.query(sql).to_dataframe()

for t in ["panel_event", "placebo_draws_strict"]:
    client.get_table(f"{DS}.{t}")
print("Tables from 05 found")

## Step 1 · L − C differences θ per date × relative day

SQL of 20, LC contrast only.

In [ ]:
OUTS = ([(f"c_{o}", f"{o}_c", f"{o}_c2", f"{o}_cb") for o in OUT_C] +
        [(f"d_{o}", f"{o}_d", f"{o}_d2", f"{o}_db") for o in OUT_D] +
        [(f"h_{h}", f"h_{h}_cum", f"h_{h}_cum", "0") for h in HZ])
CELL = "PARTITION BY event, grp, cem_id, is_large_mover ORDER BY rel_day ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW"
NF  = "CONCAT(bal_bin, '|', age_bin, '|', fs_bin, '|', rec_bin, '|', nutxo_bin)"
NFS = "CONCAT(bal_bin, '|', age_bin, '|', family, '|', rec_bin, '|', nutxo_bin)"
RS_ARM = "CASE fs_bin WHEN '2011-15' THEN 'T' WHEN '2016-19' THEN 'C' END"
UNNEST_CONTRASTS = f'''UNNEST([
    STRUCT('LC' AS contrast, CASE grp WHEN 'L' THEN 'T' WHEN 'C' THEN 'C' END AS arm, cem_id AS stratum)
  ]) AS x'''
DIMCOLS = "event, grp, cem_id, is_large_mover, rel_day, bal_bin, age_bin, family, fs_bin, rec_bin, nutxo_bin, n_keys, bal_btc"

pe_cols = ",\n    ".join(f"{a} AS {k}_c, {b} AS {k}_c2, {c} AS {k}_cb" for k, a, b, c in OUTS if not k.startswith("h_"))
pe_hz   = ",\n    ".join(f"SUM(h_{h}_n) OVER ({CELL}) AS h_{h}_cum" for h in HZ)
pe_hcol = ",\n    ".join(f"h_{h}_cum AS h_{h}_c, h_{h}_cum AS h_{h}_c2, 0 AS h_{h}_cb" for h in HZ)
K = [k for k, *_ in OUTS]
s_sum = ",\n    ".join(f"SUM({k}_c) AS {k}_c, SUM({k}_c2) AS {k}_c2, SUM({k}_cb) AS {k}_cb" for k in K)
p_piv = ",\n    ".join(
    f"SUM(IF(arm = 'T', {k}_c, 0)) AS cT_{k}, SUM(IF(arm = 'C', {k}_c, 0)) AS cC_{k}, "
    f"SUM(IF(arm = 'T', {k}_c2, 0)) AS qT_{k}, SUM(IF(arm = 'C', {k}_c2, 0)) AS qC_{k}, "
    f"SUM(IF(arm = 'T', {k}_cb, 0)) AS bT_{k}, SUM(IF(arm = 'C', {k}_cb, 0)) AS bC_{k}" for k in K)
M = "(nT > 0 AND nC > 0)"
MB = "(nT > 0 AND nC > 0 AND bT > 0 AND bC > 0)"
f_sum = ",\n  ".join(
    f"SUM(IF({M}, cT_{k}, 0)) AS T_{k}, "
    f"SUM(IF({M}, SAFE_DIVIDE(nT, nC) * cC_{k}, 0)) AS C_{k},\n  "
    f"SUM(IF({M}, qT_{k} - SAFE_DIVIDE(cT_{k} * cT_{k}, nT), 0)) AS vT_{k}, "
    f"SUM(IF({M}, POW(SAFE_DIVIDE(nT, nC), 2) * (qC_{k} - SAFE_DIVIDE(cC_{k} * cC_{k}, nC)), 0)) AS vC_{k},\n  "
    f"SUM(IF({MB}, bT_{k}, 0)) AS bT_{k}, SUM(IF({MB}, SAFE_DIVIDE(bT, bC) * bC_{k}, 0)) AS bC_{k}" for k in K)

SQL_THETA = f'''
WITH pe0 AS (
  SELECT {DIMCOLS},
    {pe_cols},
    {pe_hz}
  FROM `{DS}.panel_event`
),
pe AS (
  SELECT pe0.*, {pe_hcol} FROM pe0
),
u AS (
  SELECT pe.*, x.contrast, x.arm, x.stratum, v AS variant
  FROM pe,
  {UNNEST_CONTRASTS},
  UNNEST(IF(pe.is_large_mover, ['all'], ['all', 'no_lm'])) AS v
  WHERE x.arm IS NOT NULL
),
s AS (
  SELECT event, rel_day, contrast, variant, stratum, arm, ANY_VALUE(bal_bin) AS bal_bin,
    SUM(n_keys) AS n, SUM(bal_btc) AS b,
    {s_sum}
  FROM u
  GROUP BY event, rel_day, contrast, variant, stratum, arm
),
p AS (
  SELECT event, rel_day, contrast, variant, stratum, ANY_VALUE(bal_bin) AS bal_bin,
    SUM(IF(arm = 'T', n, 0)) AS nT, SUM(IF(arm = 'C', n, 0)) AS nC,
    SUM(IF(arm = 'T', b, 0)) AS bT, SUM(IF(arm = 'C', b, 0)) AS bC,
    {p_piv}
  FROM s
  GROUP BY event, rel_day, contrast, variant, stratum
),
f AS (
  SELECT p.*, sg FROM p, UNNEST(['all', bal_bin]) AS sg
)
SELECT event, rel_day, contrast, variant, sg AS subgroup,
  SUM(nT) AS nT_all, SUM(nC) AS nC_all,
  SUM(IF({M}, nT, 0)) AS den, SUM(IF({M}, nC, 0)) AS denC, SUM(IF({MB}, bT, 0)) AS bden,
  {f_sum}
FROM f
GROUP BY event, rel_day, contrast, variant, subgroup
'''
th = q(SQL_THETA)
ev = q(f'''SELECT event, ANY_VALUE(event_set) AS event_set, ANY_VALUE(event_date) AS event_date,
                  ANY_VALUE(censor_day) AS censor_day, ANY_VALUE(strict_ok) AS strict_ok,
                  ANY_VALUE(cb_confounded) AS cb_confounded
           FROM `{DS}.panel_event` GROUP BY event''')
ev["event_date"] = pd.to_datetime(ev.event_date)
ev = ev.sort_values("event_date").reset_index(drop=True)
print(f"theta table {len(th):,} rows; {ev.event.nunique()} dates:", ev.event_set.value_counts().to_dict())

# long table: each row = series (contrast, variant, subgroup, outcome, weight, stat, rel_day) x event
rows = []
for k in K:
    base = th[["event", "rel_day", "contrast", "variant", "subgroup"]].copy()
    den, bden = th.den.replace(0, np.nan), th.bden.replace(0, np.nan)
    mT, mC = th[f"T_{k}"] / den, th[f"C_{k}"] / den
    se = np.sqrt((th[f"vT_{k}"] + th[f"vC_{k}"]).clip(lower=0)) / den
    for stat, val in [("diff", mT - mC), ("T", mT), ("C", mC), ("se", se)]:
        rows.append(base.assign(outcome=k, weight="key", stat=stat, value=val.values))
    if not k.startswith("h_"):
        bT, bC = th[f"bT_{k}"] / bden, th[f"bC_{k}"] / bden
        for stat, val in [("diff", bT - bC), ("T", bT), ("C", bC)]:
            rows.append(base.assign(outcome=k, weight="btc", stat=stat, value=val.values))
L = pd.concat(rows, ignore_index=True)
SER = ["contrast", "variant", "subgroup", "outcome", "weight", "stat", "rel_day"]
TH = L.pivot_table(index=SER, columns="event", values="value", aggfunc="first")
TH = TH.reindex(columns=ev.event.tolist())
print("Series:", len(TH), "; matching rate (LC, all, r=0):")
m0 = th[(th.contrast == "LC") & (th.variant == "all") & (th.subgroup == "all") & (th.rel_day == 0)].set_index("event")
display((m0.den / m0.nT_all).rename("T_matched").describe().round(3))

## Step 2 · Baseline smoothing and randomization inference (helpers of 20)

In [ ]:
T0 = pd.Timestamp("2020-01-01")
tday = ((ev.event_date - T0).dt.days).to_numpy(float)
EV_IDX = {e: i for i, e in enumerate(ev.event)}
draws_all = q(f"SELECT pool, draw_id, slot, fake_date FROM `{DS}.placebo_draws_strict`")
draws_all["fake_date"] = pd.to_datetime(draws_all.fake_date)
pl = ev[ev.event_set == "placebo"].set_index("event_date").event
POOL_EVENTS, DRAWS = {}, {}
for pool in POOLS:
    d = draws_all[draws_all.pool == pool]
    dates = sorted(d.fake_date.unique())
    missing = [x for x in dates if x not in pl.index]
    assert not missing, f"dates of pool {pool} not found in panel_event: {missing}"
    POOL_EVENTS[pool] = [pl[x] for x in dates]
    pos = {x: j for j, x in enumerate(dates)}
    D = np.zeros((d.draw_id.nunique(), len(dates)))
    for r in d.itertuples():
        D[r.draw_id - 1, pos[r.fake_date]] += 1
    DRAWS[pool] = D / D.sum(1, keepdims=True)
    print(f"pool {pool}: {len(dates)} dates, {len(D)} draws")

def smoother(pool, h, mask=None):
    idx = np.array([EV_IDX[e] for e in POOL_EVENTS[pool]])
    tp = tday[idx]
    ok = np.ones(len(idx), bool) if mask is None else mask
    lo = tp[ok].min() - 60
    S = np.zeros((len(ev), len(idx)))
    for i in range(len(ev)):
        w = np.exp(-0.5 * ((tp - tday[i]) / h) ** 2) * ok
        self_ = idx == i
        w[self_] = 0
        if tday[i] < lo:                           # before the pool: pool mean
            w = ok.astype(float); w[self_] = 0
            S[i] = w / w.sum(); continue
        X = np.column_stack([np.ones_like(tp), tp - tday[i]])
        A = X.T @ (X * w[:, None])
        S[i] = np.linalg.solve(A, (X * w[:, None]).T)[0]
    return S

def effects(Th, pool, h):
    idx = [EV_IDX[e] for e in POOL_EVENTS[pool]]
    Y = Th.to_numpy(float)
    P = Y[:, idx]
    B = np.full_like(Y, np.nan)
    nanmask = np.isnan(P)
    keys = pd.Series([m.tobytes() for m in nanmask])
    for key, rows_ in keys.groupby(keys).groups.items():
        rows_ = np.array(list(rows_))
        ok = ~nanmask[rows_[0]]
        if ok.sum() < 5:
            continue
        S = smoother(pool, h, ok)
        B[rows_] = Y[rows_] - np.nan_to_num(P[rows_]) @ S.T
    R = B[:, idx]
    return pd.DataFrame(B, index=Th.index, columns=Th.columns), R

def ri_p(beta, R):
    b = np.abs(np.asarray(beta, float))[:, :, None]
    r = np.abs(R)[:, None, :]
    cnt = np.sum(r >= b - 1e-15, axis=2)
    n = np.sum(~np.isnan(R), axis=1)[:, None]
    p = (1 + cnt) / (1 + n)
    p[np.isnan(np.asarray(beta, float))] = np.nan
    return p

def ri_ci(beta, R, a=0.10):
    lo_q, hi_q = np.nanquantile(R, 1 - a / 2, axis=1), np.nanquantile(R, a / 2, axis=1)
    return beta - lo_q[:, None], beta - hi_q[:, None]
print("Helpers ready")

## Step 3 · Pooled estimates by subset

Single events (A, B): p-values and 90% intervals from leave-one-out residuals, as in 20. Pooled subsets (C–F): null = mean residual of k dates drawn without replacement from the pool, 1,000 draws with a fixed seed, shared across bandwidths. F uses the draws of 20.

Regression check: the mean β of F equals `est_20_pooled.csv`.

In [ ]:
diff = TH.xs("diff", level="stat")
lev_T = TH.xs("T", level="stat").reindex(diff.index)
MAIN = ev[ev.event_set == "main"].event.tolist()
for nm, evs in SUBSETS.items():
    assert set(evs) <= set(MAIN), f"{nm}: {set(evs) - set(MAIN)} are not main events"

DK = {}
for pool in POOLS:
    npool = len(POOL_EVENTS[pool])
    for k in sorted({len(v) for v in SUBSETS.values()}):
        rng = np.random.default_rng(SEED + 100 * k + (0 if pool == "main" else 1))
        D = np.zeros((N_DRAWS, npool))
        for i in range(N_DRAWS):
            D[i, rng.choice(npool, size=k, replace=False)] = 1.0 / k
        DK[(pool, k)] = D

rows, KEEP = [], {}
for pool in POOLS:
    for h in BANDWIDTHS:
        B, R = effects(diff, pool, h)
        KEEP[(pool, h)] = (B, R)
        for nm, evs in SUBSETS.items():
            k = len(evs)
            mb = B[evs].mean(axis=1).to_numpy()
            if k == 1:
                p = ri_p(B[evs].to_numpy(), R)[:, 0]
                lo, hi = ri_ci(B[evs].to_numpy(), R)
                lo, hi, null_mean = lo[:, 0], hi[:, 0], np.nanmean(R, axis=1)
            else:
                D = DRAWS[pool] if nm == "F_all_six" else DK[(pool, k)]
                null = np.nan_to_num(R) @ D.T
                p = (1 + np.sum(np.abs(null) >= np.abs(mb)[:, None] - 1e-15, axis=1)) / (1 + null.shape[1])
                lo, hi = mb - np.quantile(null, 0.95, axis=1), mb - np.quantile(null, 0.05, axis=1)
                null_mean = null.mean(axis=1)
            rows.append(pd.DataFrame({"subset": nm, "events": "+".join(evs), "k": k, "pool": pool, "h": h,
                                      "beta_mean": mb, "p_ri": p, "ci90_lo": lo, "ci90_hi": hi, "null_mean": null_mean,
                                      "level_T_mean": lev_T[evs].mean(axis=1).to_numpy()}, index=diff.index).reset_index())
sub = pd.concat(rows, ignore_index=True)
sub = sub[sub.rel_day.isin(REPORT_DAYS)]
sub.to_csv(f"{OUT_DIR}/est_20b_subsets.csv", index=False)

MAINSPEC = (sub.contrast == "LC") & (sub.variant == "all") & (sub.subgroup == "all") & (sub.weight == "key") & (sub.pool == "main") & (sub.h == H_MAIN)
f30 = sub[MAINSPEC & (sub.subset == "F_all_six") & (sub.outcome == "c_y1_strict") & (sub.rel_day == 30)].iloc[0]
print(f"Reproducing 20: pooled beta of F = {f30.beta_mean:.10f} (20: {REF_20_POOLED:.10f}), p = {f30.p_ri:.4f}")
assert abs(f30.beta_mean - REF_20_POOLED) < 5e-9, "the pooled effect of 20 was not reproduced"
show = sub[MAINSPEC & sub.outcome.isin(["c_y1_strict", "c_y1_strict_held", "h_y1s_50", "c_y0"])]
display(show.pivot_table(index=["outcome", "rel_day", "subset"], values=["beta_mean", "ci90_lo", "ci90_hi", "p_ri"], aggfunc="first")
        .reindex(columns=["beta_mean", "ci90_lo", "ci90_hi", "p_ri"]).round(5))

## Step 4 · Calendar-time figure data (supplementary materials)

x = date. Grey points: raw L − C θ on the 29 placebo dates; curve: local linear fit (main pool, h = 90 days); band: fit + 5–95% quantiles of the leave-one-out residuals; colored points: the six main events (Willow and Google filled) and non-quantum news days. β is the vertical distance from an event point to the curve.

In [ ]:
def curve(th_row, pool, h, ts):
    idx = np.array([EV_IDX[e] for e in POOL_EVENTS[pool]])
    tp, yp = tday[idx], th_row.reindex(POOL_EVENTS[pool]).to_numpy(float)
    ok = ~np.isnan(yp); tp, yp = tp[ok], yp[ok]
    out_ = []
    for t in ts:
        w = np.exp(-0.5 * ((tp - t) / h) ** 2)
        X = np.column_stack([np.ones_like(tp), tp - t])
        out_.append(np.linalg.solve(X.T @ (X * w[:, None]), (X * w[:, None]).T @ yp)[0])
    return np.array(out_)

B90, R90 = KEEP[("main", H_MAIN)]
pool_dates = ev.set_index("event").event_date.reindex(POOL_EVENTS["main"])
grid = pd.date_range(pool_dates.min() - pd.Timedelta(days=30), pool_dates.max() + pd.Timedelta(days=30), freq="7D")
tg = ((grid - T0).days).to_numpy(float)
cal_rows, curve_rows = [], []
for r_ in REPORT_DAYS:
    key_ = ("LC", "all", "all", "c_y1_strict", "key", r_)
    th = diff.loc[key_]
    b = B90.loc[key_]
    rk = R90[diff.index.get_loc(key_)]
    q05, q95 = np.nanquantile(rk, 0.05), np.nanquantile(rk, 0.95)
    for e in ev.itertuples():
        if pd.isna(th.get(e.event, np.nan)): continue
        cal_rows.append({"rel_day": r_, "event": e.event, "event_set": e.event_set, "event_date": e.event_date,
                         "theta": th[e.event], "baseline": th[e.event] - b[e.event], "beta": b[e.event],
                         "in_pool_main": e.event in POOL_EVENTS["main"]})
    fit = curve(th, "main", H_MAIN, tg)
    curve_rows.append(pd.DataFrame({"rel_day": r_, "date": grid, "fit": fit, "band_lo": fit + q05, "band_hi": fit + q95}))
cal = pd.DataFrame(cal_rows)

# level paths of the main events (treated T, CEM-weighted control C), for the Kaplan-Meier figure in the supplementary materials
lev_rows = []
for o in ["h_y1s_50", "c_y1_strict", "c_y1_strict_held"]:
    for stat in ["T", "C"]:
        s_ = TH.xs(("LC", "all", "all", o, "key", stat), level=["contrast", "variant", "subgroup", "outcome", "weight", "stat"])
        for e in MAIN:
            lev_rows.append(pd.DataFrame({"outcome": o, "stat": stat, "event": e, "rel_day": s_.index, "value": s_[e].values}))
pd.concat(lev_rows, ignore_index=True).to_csv(f"{OUT_DIR}/est_20b_levels.csv", index=False)
crv = pd.concat(curve_rows, ignore_index=True)
pd.concat([cal.assign(kind="date"), crv.rename(columns={"date": "event_date"}).assign(kind="curve")],
          ignore_index=True).to_csv(f"{OUT_DIR}/est_20b_calendar.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=False)
for ax, r_ in zip(axes, REPORT_DAYS):
    c = crv[crv.rel_day == r_]; d = cal[cal.rel_day == r_]
    ax.fill_between(c.date, c.band_lo * 100, c.band_hi * 100, color="0.88", lw=0)
    ax.plot(c.date, c.fit * 100, color="0.35", lw=1.2)
    pp = d[d.in_pool_main]
    ax.scatter(pp.event_date, pp.theta * 100, s=14, color="0.45", zorder=3, label="placebo dates")
    mm = d[d.event_set == "main"]
    for e in mm.itertuples():
        hi_att = e.event in ("willow", "google")
        ax.scatter(e.event_date, e.theta * 100, s=55 if hi_att else 36, zorder=4,
                   color="#D55E00" if hi_att else "#E69F00", marker="D" if hi_att else "o")
        ax.annotate(e.event, (e.event_date, e.theta * 100), fontsize=7, xytext=(3, 3), textcoords="offset points")
    nq = d[d.event_set == "nonquantum_news"]
    ax.scatter(nq.event_date, nq.theta * 100, s=30, color="#0072B2", marker="s", zorder=4, label="non-quantum news days")
    ax.axhline(0, color="k", lw=0.5)
    ax.set_title(f"Exposed − control, cumulative y1 strict share, day +{r_}", fontsize=9)
    ax.set_ylabel("percentage points")
    ax.tick_params(axis="x", labelrotation=30, labelsize=7)
    for s_ in ["top", "right"]: ax.spines[s_].set_visible(False)
axes[0].legend(fontsize=7, frameon=False)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_20b_calendar.png", dpi=200); plt.show()

In [ ]:
report = {"generated_at": dt.datetime.now(dt.timezone.utc).isoformat(), "bq_usd_estimate": round(SPENT["usd"], 3),
          "subsets": SUBSETS, "pool_sizes": {p: len(v) for p, v in POOL_EVENTS.items()},
          "reproduce_20": {"beta_mean": float(f30.beta_mean), "ref": REF_20_POOLED, "p_ri": float(f30.p_ri)},
          "main_spec": show.astype(str).to_dict(orient="records")}
with open(f"{OUT_DIR}/build_report_20b.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2, default=str)
with zipfile.ZipFile("out20b.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for fn in sorted(os.listdir(OUT_DIR)):
        z.write(os.path.join(OUT_DIR, fn), arcname=f"out20b/{fn}")
print("Packed out20b.zip:", sorted(os.listdir(OUT_DIR)))
try:
    from google.colab import files
    files.download("out20b.zip")
except Exception:
    print("Not in Colab: use out20b.zip directly")